# H2 掴む：形・目印・閉じ方・向き

**ねらい**：箱をうまく掴めるかどうかは、力の大きさの前に、次の 4 つで決まる。作品を作るときに 1 つずつ踏んだ。
| # | 決めたこと | 作品の値 | わざと失敗 |
|---|---|---|---|
| ① | ぶつかり判定のための、形の包み方 | 細かく分ける（0.0） | `convexify` |
| ② | 目印（指先）から箱の中心までのずれ | 1.9 cm | `jaw_center` |
| ③ | 指を速く閉じる 1 段目の止め所 | 0.5 | `pre_close` |
| ④ | 箱の向きに、指を合わせるか | 合わせる | `box_world` |

どれも、値を書き換えて違いを確かめる。

## 1. 準備

In [ ]:
import os, time
if "google.colab" in str(get_ipython()):
    !apt-get -qq install -y libosmesa6 fonts-noto-cjk > /dev/null
    !pip -q install genesis-world==1.4.3
    if not os.path.exists("SO-ARM100"):
        !git clone -q https://github.com/TheRobotStudio/SO-ARM100.git && cd SO-ARM100 && git checkout -q a758567
    MODEL = "/content/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
else:   # 手元の Docker（答え合わせ用）
    MODEL = "/work/third_party/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
import numpy as np, matplotlib.pyplot as plt, genesis as gs
# グラフの日本語：Colab は matplotlib を先に読み込んでいるので、入れたフォントを直接登録する（名前だけ指定しても見つからない）
from matplotlib import font_manager
for _f in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
    if os.path.exists(_f):
        font_manager.fontManager.addfont(_f)
plt.rcParams["font.family"] = ["Noto Sans CJK JP", "DejaVu Sans"]

## 2. 場面：形の包み方が違う SO-101 を 2 台
- **細**：形を細かい凸の部品に分けて包む（`decompose_robot_error_threshold=0.0`。作品）
- **粗**：Genesis の既定（部品ごとに 1 つの凸で包む）

2 台は 50 cm 離して、それぞれの前（根元から 22 cm）に箱を置く。組み立ては 1 回だけ。

In [ ]:
import math
OFF = np.array([-0.0079, -0.000218, -0.0981]); FOLD = np.array([0.0, -1.7, 1.6, 1.2, 0.0, 0.0])
OPEN, CLOSE = 1.2, -0.17
def rot(q):
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])
def quat_of(R):
    t = np.trace(R)
    if t > 0:
        S = math.sqrt(t + 1.0) * 2; q = [0.25 * S, (R[2, 1] - R[1, 2]) / S, (R[0, 2] - R[2, 0]) / S, (R[1, 0] - R[0, 1]) / S]
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        S = math.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2]) * 2; q = [(R[2, 1] - R[1, 2]) / S, 0.25 * S, (R[0, 1] + R[1, 0]) / S, (R[0, 2] + R[2, 0]) / S]
    elif R[1, 1] > R[2, 2]:
        S = math.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2]) * 2; q = [(R[0, 2] - R[2, 0]) / S, (R[0, 1] + R[1, 0]) / S, 0.25 * S, (R[1, 2] + R[2, 1]) / S]
    else:
        S = math.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1]) * 2; q = [(R[1, 0] - R[0, 1]) / S, (R[0, 2] + R[2, 0]) / S, (R[1, 2] + R[2, 1]) / S, 0.25 * S]
    q = np.array(q); return q / np.linalg.norm(q)
def Ry(a): return np.array([[math.cos(a), 0, math.sin(a)], [0, 1, 0], [-math.sin(a), 0, math.cos(a)]])
def Rz(a): return np.array([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0], [0, 0, 1]])

gs.init(backend=gs.cpu, logging_level="warning")
sc = gs.Scene(sim_options=gs.options.SimOptions(dt=0.01, substeps=10),
              rigid_options=gs.options.RigidOptions(box_box_detection=True, friction_cone=gs.friction_cone.elliptic, impratio=10.0),
              show_viewer=False)
sc.add_entity(gs.morphs.Plane())
RB, Y2 = 0.22, 0.5
fine = sc.add_entity(gs.morphs.MJCF(file=MODEL, pos=(0, 0, 0), decompose_robot_error_threshold=0.0), material=gs.materials.Rigid(friction=1.0))
coarse = sc.add_entity(gs.morphs.MJCF(file=MODEL, pos=(0, Y2, 0)), material=gs.materials.Rigid(friction=1.0))
mat, look = gs.materials.Rigid(friction=1.0, rho=1000.0), gs.surfaces.Default(color=(0.72, 0.55, 0.36))
box_f = sc.add_entity(gs.morphs.Box(size=(0.03,) * 3, pos=(RB, 0, 0.015)), material=mat, surface=look)
box_c = sc.add_entity(gs.morphs.Box(size=(0.03,) * 3, pos=(RB, Y2, 0.015)), material=mat, surface=look)
cam = sc.add_camera(res=(768, 432), pos=(0.21, -0.30, 0.13), lookat=(0.21, 0.0, 0.04), fov=45, GUI=False)
t0 = time.time(); sc.build(); print(f"build {time.time() - t0:.0f} s")

class Hand:
    """真下向きの手先で、自分の前の箱を掴む（作品の共通部品 Arm を、この演習用に小さくしたもの）"""
    def __init__(self, ent, base_y, box):
        self.e, self.g, self.box, self.y = ent, ent.get_link("gripper"), box, base_y
        self.fixed, self.moving = self.g.idx, ent.get_link("moving_jaw_so101_v1").idx
        ent.set_dofs_position(np.zeros(6)); self.R0 = rot(self.g.get_quat().cpu().numpy())
        self.dyaw, self.cur = 0.0, None
    def Rd(self): return Rz(self.dyaw) @ Ry(math.pi / 2) @ self.R0          # 指を dyaw だけ回して、真下へ
    def ik(self, p):
        R = self.Rd()
        return self.e.inverse_kinematics(link=self.g, pos=p - R @ OFF, quat=quat_of(R), init_qpos=self.e.get_dofs_position(),
                                         max_solver_iters=100).cpu().numpy()
    def tip(self): return self.g.get_pos().cpu().numpy() + rot(self.g.get_quat().cpu().numpy()) @ OFF
    def jaws(self):
        c = self.e.get_contacts(with_entity=self.box)
        la, lb = c["link_a"].cpu().numpy(), c["link_b"].cpu().numpy()
        f = np.linalg.norm(c["force_a"].cpu().numpy(), axis=-1)
        if "valid_mask" in c: f = f * c["valid_mask"].cpu().numpy()
        return float((f * ((la == self.fixed) | (lb == self.fixed))).sum()), float((f * ((la == self.moving) | (lb == self.moving))).sum())
    def set_grip(self, tau):
        lim = np.full(6, 3.35); lim[5] = tau; self.e.set_dofs_force_range(-lim, lim)
    def go(self, p, grip, n, every=None):
        start = self.cur
        for k in range(n):
            s = min(1.0, (k + 1) / (0.7 * n)); s = 3 * s * s - 2 * s ** 3
            if k % 5 == 0 or k == n - 1:
                q = self.ik(start + (np.asarray(p, float) - start) * s); q[5] = grip; self.e.control_dofs_position(q)
            sc.step()
            if every: every()
        self.cur = np.asarray(p, float)

H = {"細": Hand(fine, 0.0, box_f), "粗": Hand(coarse, Y2, box_c)}

def box_state(h):
    p = h.box.get_pos().cpu().numpy(); q = h.box.get_quat().cpu().numpy()
    tilt = math.degrees(math.acos(max(-1.0, min(1.0, 1 - 2 * (q[1] ** 2 + q[2] ** 2)))))
    return p, tilt

def trial(h, tau=0.4, mass=0.15, jaw=0.019, pre=0.5, box_yaw=0.0, align=True, lift=0.04):
    """h で、自分の前の箱を掴んで lift だけ持ち上げる。結果の dict を返す。
    jaw：目印から箱の中心までのずれ／pre：速く閉じる 1 段目の止め所／box_yaw：箱の向き [°]／align：指を箱の向きに合わせるか"""
    sc.reset()
    for x in H.values():
        x.e.set_dofs_position(FOLD); x.e.control_dofs_position(FOLD)
    h.box.set_mass(mass)
    h.box.set_quat(np.array([math.cos(math.radians(box_yaw) / 2), 0, 0, math.sin(math.radians(box_yaw) / 2)]))
    h.dyaw = ((math.radians(box_yaw) + math.pi / 4) % (math.pi / 2) - math.pi / 4) if align else 0.0   # 四角は 90° ごとに同じ → ±45° に
    c = np.array([RB, h.y, 0.0]) - jaw * np.array([math.cos(h.dyaw), math.sin(h.dyaw), 0])          # 目印は、指が閉じる向きに jaw だけ手前
    h.cur = np.array([c[0], c[1], 0.08]); q = h.ik(h.cur); q[5] = OPEN; h.e.set_dofs_position(q); h.e.control_dofs_position(q)
    h.set_grip(3.35)
    for _ in range(30): sc.step()
    peak = [0.0]
    def watch(): peak[0] = max(peak[0], min(h.jaws()))
    h.go([c[0], c[1], 0.02], OPEN, 100, watch)
    h.go(h.cur, pre, 40, watch)                       # 1 段目：力の上限を大きくしたまま、pre まで速く閉じる
    h.set_grip(tau); h.go(h.cur, CLOSE, 200, watch)    # 2 段目：力の上限を tau に絞って閉じきる
    sq = min(h.jaws())
    cc = h.e.get_contacts(with_entity=h.box)                                   # 指が箱に触れている所の高さ（閉じきったとき）
    contact_heights = sorted(round(float(z) * 100, 2) for z in cc["position"].cpu().numpy().reshape(-1, 3)[:, 2])
    grip_q = float(h.e.get_dofs_position()[5])
    h.go([c[0], c[1], 0.02 + lift], CLOSE, 80, watch); h.go(h.cur, CLOSE, 50, watch)
    p, tilt = box_state(h)
    trial.contacts = contact_heights
    return {"挟む力": sq, "閉じる間の最大": peak[0], "箱の高さ": p[2], "傾き": tilt, "上がった": p[2] > 0.015 + lift * 0.5, "指の角度": grip_q}

def show(h=None, title=""):
    sc.visualizer.update(force=True)
    if h is not None:
        cam.set_pose(pos=(0.21, h.y - 0.30, 0.13), lookat=(0.21, h.y, 0.04))
    plt.figure(figsize=(7, 4)); plt.imshow(cam.render(rgb=True)[0]); plt.axis("off"); plt.title(title); plt.show()

def say(label, r):
    print(f"{label:22s} 挟む力 {r['挟む力']:5.2f} N   閉じる間の最大 {r['閉じる間の最大']:5.2f} N   指の角度 {r['指の角度']:4.2f}   箱の高さ {r['箱の高さ'] * 100:4.1f} cm   傾き {r['傾き']:4.1f}°")

## 3. ① 形の包み方
ぶつかり判定は、計算を速くするため、部品の形を **凸の形**（へこみのない形）で包んで行う。
SO-101 の固定側の指は、手首と 1 つの部品になっている。1 つの凸で包むと、**指と手首のあいだのすき間が埋まる**。

**予想**：粗く包んだ腕は、箱を掴める？

In [ ]:
for name in ["細", "粗"]:
    r = trial(H[name]); say(f"{name}く包む", r); print("    指が箱に触れている高さ [cm]：", trial.contacts, "（箱は 0〜3 cm）")
    show(H[name], "fine" if name == "細" else "coarse")

**期待される出力**（手元の実測）
```
細く包む   挟む力 5.36 N   閉じる間の最大  7.73 N   指の角度 0.20   箱の高さ 5.5 cm
    指が箱に触れている高さ [cm]： [1.6, 2.12, 2.13, 3.0]
粗く包む   挟む力 5.36 N   閉じる間の最大 10.07 N   指の角度 0.30   箱の高さ 1.5 cm
    指が箱に触れている高さ [cm]： [2.98, 2.99]
```
**挟む力は同じなのに、粗く包んだ腕は持ち上がらない**。触れている所を見ると、箱の上の角（3 cm）だけ。埋まったすき間に邪魔されて指が下りきれず（指の角度 0.30）、角をつまんでいる。
挟む力の数字だけでは分からない。**触れている場所** を見る。

## 4. ② 目印から箱の中心までのずれ
目印 `gripperframe` は、**固定側の指の内側の先端**（指の真ん中ではない）。箱の中心は、目印から 1.9 cm 先（指が閉じる向き）。
**予想**：目印を箱の中心に置く（`jaw=0`）と？

In [ ]:
for jaw in [0.019, 0.0]:
    r = trial(H["細"], jaw=jaw); say(f"jaw = {jaw * 100:.1f} cm", r)
show(H["細"], "jaw = 0")

## 5. ③ 指を閉じる 1 段目の止め所
力の上限を小さくすると、指はゆっくりしか閉じない（H1c）。そこで作品は 2 段階で閉じる：**① 力の上限を大きくしたまま、触れる手前（角度 0.5）まで速く閉じる → ② 力の上限を τ に絞って閉じきる**。
3 cm の箱に指が触れるのは、角度 0.20〜0.24。
**予想**：1 段目を 0.3、0.25（触れる直前）、0.2（触れた後）にすると、閉じる間の最大の力は？ 段ボールが潰れるのは 15 N。

In [ ]:
for pre in [0.5, 0.3, 0.25, 0.2]:
    r = trial(H["細"], pre=pre); say(f"1 段目 {pre}", r)

**期待される出力**：最大の力は 0.5 で 7.7 N、0.3 で 8.3 N、0.25 で 13.7 N、0.2 で 66 N。触れる直前まで大きな力のまま閉じると、勢いで当たる。最後に挟む力（4.8〜5.4 N）はほとんど変わらないので、**閉じる途中を見ないと気づかない**。

## 6. ④ 箱の向きに指を合わせる
箱が 30° 回って置かれていたら、指も回して、箱の面を挟む。四角は 90° ごとに同じ形なので、回す角度は ±45° に収める。
**予想**：指を合わせないと（`align=False`）、どこを挟む？

In [ ]:
for align in [True, False]:
    r = trial(H["細"], box_yaw=30, align=align); say(f"30° の箱・指を{'合わせる' if align else '合わせない'}", r)
    show(H["細"], f"指を{'合わせる' if align else '合わせない'}")

**期待される出力**：合わせると 5.5 cm まで上がる。合わせないと、指は箱の角に当たり、挟む力 0 で上がらない。

## 確かめる問い
1. 粗く包んだ腕が掴めないのはなぜ？ 画面で何が見える？
2. 目印を箱の中心に置くと、何が起きる？
3. 指を閉じる 1 段目を、触れる手前の 0.5 で止めるのはなぜ？

<details><summary>答え</summary>

1. 固定側の指は手首と 1 つの部品。1 つの凸で包むと、指と手首のあいだのすき間が埋まり、指が下りきれない。箱の上の角だけをつまむので、持ち上げると抜ける（見た目の形は同じなので、画面ではすき間があるように見える）
2. 目印は固定側の指の先端なので、そこを箱の中心に置くと、固定側の指が箱の上に下りる
3. 触れる直前まで大きな力のまま閉じると、勢いで当たって、閉じる間の力が箱の強さ（段ボール 15 N）を超える。最後の挟む力は変わらないので、終わった後の値では気づけない
</details>